In [ ]:
import torch
import numpy as np
import matplotlib.pyplot as plt
import random
from pathlib import Path

from train_vision import VisionEncoder 

def run_model_acceptance_test(ckpt_path):
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    model = VisionEncoder().to(device)
    model.load_state_dict(torch.load(ckpt_path, map_location=device))
    model.eval() 

    # load dataset
    data_dir = Path("../vision_nn_dataset")
    images_mmap = np.load(data_dir / "images.npy", mmap_mode='r')
    coords_mmap = np.load(data_dir / "coords.npy", mmap_mode='r')

    # randomly draw 16 samples
    num_samples = len(images_mmap)
    sampled_indices = random.sample(range(num_samples), 16)

    fig, axs = plt.subplots(4, 4, figsize=(14, 14))
    fig.suptitle("Vision Encoder Accuracy Audit: Ground Truth (O) vs Prediction (+)", fontsize=16)

    total_pixel_errors = []

    with torch.no_grad():
        for i, idx in enumerate(sampled_indices):
            ax = axs.flatten()[i]
            
            img_raw = images_mmap[idx]
            gt_coord = coords_mmap[idx]
            
            # (1, 3, 128, 128)
            img_tensor = torch.from_numpy(img_raw.copy()).permute(2, 0, 1).float() / 255.0
            img_tensor = img_tensor.unsqueeze(0).to(device)
            
            pred_coord_tensor, _ = model(img_tensor)
            pred_coord = pred_coord_tensor.cpu().numpy()[0]
            
            ax.imshow(img_raw)

            entities = ["Prey", "Pred", "Shelter", "Land1", "Land2", "Land3"]
            colors = ["cyan", "magenta", "yellow", "orange", "orange", "orange"]

            for e_idx in range(6):
                # [-1, 1] coordinates
                gx, gy = gt_coord[e_idx*2], gt_coord[e_idx*2 + 1]
                px, py = pred_coord[e_idx*2], pred_coord[e_idx*2 + 1]
                color = colors[e_idx]
                
                # Real L2 distances
                dist_norm = np.sqrt((gx - px)**2 + (gy - py)**2)
                pixel_error = (dist_norm / 2.0) * 128.0
                total_pixel_errors.append(pixel_error)
                
                g_pix_x, g_pix_y = np.round((gx + 1) / 2 * 128), np.round((gy + 1) / 2 * 128)
                p_pix_x, p_pix_y = np.round((px + 1) / 2 * 128), np.round((py + 1) / 2 * 128)
                
                # GT
                ax.plot(g_pix_x, g_pix_y, marker='o', markersize=14, 
                        markeredgecolor=color, markerfacecolor='none', markeredgewidth=2, alpha=0.5)
                # Prediction
                ax.plot(p_pix_x, p_pix_y, color=color, marker='+', markersize=12, markeredgewidth=2)
                
            ax.set_title(f"ID: {idx}", fontsize=9)
            ax.axis('off')

    plt.tight_layout()
    plt.show()

    mean_err = np.mean(total_pixel_errors)
    max_err = np.max(total_pixel_errors)
    print(f"\n{'='*50}")
    print(f"🎯 16 samples x 6 entities = 96 coordinates)")
    print(f"Mean Pixel Error: {mean_err:.3f} Pixels")
    print(f"Max Pixel Error: {max_err:.3f} Pixels")
    print(f"{'='*50}\n")

run_model_acceptance_test(ckpt_path = "../vision_nn_ckpt/best_model.pth")